# **What Makes a Mobile App Successful?**

## 1. Project Overview
- A hypothetical product team is planning a new Android app and wants to understand the Google Play market
- Goal: find which app characteristics are linked to success
- My role: the full cycle: cleaning, analysis and conclusions
- Output: a notebook on GitHub + a README with key charts
- Stack: Python, pandas, NumPy, matplotlib, seaborn

## 2. Business Problem
A team with a limited budget needs to choose a **category** and a **monetization model** (free / paid) with better chances of getting installs and good ratings.

## 3. Analytical Questions
1. Which categories are the most competitive (many apps), and which have high demand (many installs per app)?
2. Which categories have the highest ratings?
3. Do free and paid apps differ in installs and ratings?
4. What do reviews say: which categories get the most negative feedback?

## 4. Data and Approach
**Data:** [Google Play Store Apps on Kaggle](https://www.kaggle.com/datasets/lava18/google-play-store-apps) (2018): ~10,800 apps and their user reviews.

**Approach:** cleaned the data; for demand, used the median number of installs and the share of apps with 1M+ installs; for quality, the share of apps rated 4.5+ among apps with 100+ reviews; tested the Free / Paid difference with a chi-square test.

**Limitations:** the data is old (2018) and is a sample, not the whole store; Installs is a lower bound; there is no revenue data. The analysis shows associations, not causes.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
# Load data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


apps_raw = pd.read_csv('/content/drive/MyDrive/Google Play Store Apps/googleplaystore.csv')
apps = apps_raw.copy()
reviews_raw = pd.read_csv('/content/drive/MyDrive/Google Play Store Apps/googleplaystore_user_reviews.csv')
reviews = reviews_raw.copy()

## 5. Data Cleaning

In [ ]:
# Remove a broken row with shifted columns
apps = apps[~(apps["Size"].str[-1] == "+")]

print(apps.shape)

In [ ]:
# Clean Size: convert to MB, "Varies with device" to NaN

def to_megabyte(value):
  if value[-1] == "k" :
    mb = float(value[:-1])/1024
    return round(mb,3)
  elif value[-1] == "M":
    mb = float(value[:-1])
    return round(mb,3)
  else:
    return np.nan


apps["Size_num"] = apps["Size"].apply(to_megabyte)

# Check: the number of NaN values matches the number of "Varies with device"
(apps["Size"] == "Varies with device").sum() == apps["Size_num"].isna().sum()


In [ ]:
# Clean Installs: remove commas and "+"

apps["Installs_num"] = apps["Installs"].str.replace(",","").str.replace("+","")
apps["Installs_num"] = apps["Installs_num"].astype('int64')

# Check: each original value and its number
apps[["Installs","Installs_num"]].drop_duplicates().sort_values("Installs_num")

In [ ]:
# Clean Price: remove "$"

apps["Price_num"] = apps["Price"].str.replace("$","").astype('float')

# Check: each original price and its number
apps[["Price", "Price_num"]].drop_duplicates().sort_values("Price_num")


In [ ]:
# Convert Last Updated to a date
apps["Last_Updated_date"]= pd.to_datetime(apps["Last Updated"])

# Check: date range
print(apps["Last_Updated_date"].min())
print(apps["Last_Updated_date"].max())


In [ ]:
# Convert Reviews to numbers

apps["Reviews_num"] = apps["Reviews"].astype('int64')


# Check
apps["Reviews"].isna().sum()

In [ ]:
# Remove duplicates: first full duplicates, then keep the record with the most reviews for each app
apps = apps.drop_duplicates()
apps = apps.sort_values(["App","Reviews_num"], ascending=[True,False])
apps = apps.drop_duplicates(subset = "App", keep='first')

# Check: one row per app
print(apps.duplicated(subset=["App"]).sum())
apps.shape[0] ==apps["App"].nunique()


In [ ]:
# Remove the row with missing Type (row 9148: no other data in it)
apps = apps[~apps["Type"].isna()]

# Check
apps[apps["Type"].isna()]

In [ ]:
# Check: ratings are between 1 and 5
print(apps["Rating"].min())
print(apps["Rating"].max())


In [ ]:
# Reset index
apps = apps.reset_index(drop=True)

In [ ]:
# Keep only the needed columns
apps = apps.drop(["Size","Reviews","Installs","Price","Last Updated"],axis = 1)

In [ ]:
# Before / after cleaning
summary = pd.DataFrame({
    "Metric": ["Rows",
               "Columns",
               "Full duplicates",
               "Duplicates by App",
               "Missing Rating"],
    "Before":[apps_raw.shape[0],
              apps_raw.shape[1],
              apps_raw.duplicated().sum(),
              apps_raw["App"].duplicated().sum(),
              apps_raw["Rating"].isna().sum()],
    "After": [apps.shape[0],
              apps.shape[1],
              apps.duplicated().sum(),
              apps["App"].duplicated().sum(),
              apps["Rating"].isna().sum()]
    })

summary


## 6. Analysis

**Question 1. Which categories are the most competitive (many apps), and which have high demand (many installs per app)?**

In [ ]:
category_value = apps["Category"].value_counts().sort_values(ascending = True)


In [ ]:
categories = category_value.index

In [ ]:
c_values = category_value.values

In [ ]:
plt.figure(figsize=(10, 11))
plt.barh(categories,c_values)
plt.title("Number of apps per category")
plt.xlabel("Number of apps")
plt.show()


In [ ]:
share_category = (apps["Category"]).value_counts(normalize = True)*100
print(f"Top 3 categories: {round(share_category.head(3).sum())}% of all apps")
print(round(share_category.head(3)))


The most competitive categories: FAMILY (19%), GAME (10%) and TOOLS (9%).
Together, these three categories make up 38% of all apps.

In [ ]:
# Typical number of installs per category (median)
installs_median = apps.groupby("Category")["Installs_num"].median()

# Share of apps with 1M+ installs per category
apps["Installs_1M_plus"] = apps["Installs_num"] >= 1000000.0
top_share = round(apps.groupby("Category")["Installs_1M_plus"].mean()*100)

category_demand = pd.DataFrame({
    "Median":installs_median,
    "Top_share":top_share
    })


category_demand.sort_values(["Median","Top_share"],ascending=[False,False])

In [ ]:
# Share of apps with 1M+ installs across all apps
(apps["Installs_1M_plus"]).value_counts(normalize=True)*100

In 7 categories, a typical app has 1M+ installs.

ENTERTAINMENT leads: 84% of its apps have 1M+ installs, compared with 35% across all apps.

In [ ]:
# Category rank by number of apps (1 = the most apps)
category_value.rank(ascending=False)

**Conclusion 1:**

ENTERTAINMENT has the largest share of apps with 1M+ installs: 84% vs 35% across all apps.
Competition in this category is low: it ranks 24th of 33 by number of apps, so ENTERTAINMENT looks promising. However, the category has few apps, so this share is less stable.

FAMILY has the most apps, but only 30% of them reach 1M+ installs, below the overall 35%.

**Question 2. Which categories have the highest ratings?**

In [ ]:
# Choosing a threshold: ratings based on few reviews are unreliable
apps["Reviews_num"].describe()

In [ ]:
# Proposed threshold: 100 reviews. How many rated apps would we exclude?
reviews_less_100 = apps[apps["Reviews_num"] < 100].dropna(subset = ["Rating"])
len(reviews_less_100)


In [ ]:
# Check the 100-review threshold
print(f"Rating 5.0 and <100 reviews: {((apps['Rating'] == 5.0) & (apps['Reviews_num'] < 100)).sum()} apps")
print(f"Rating 5.0 and >=100 reviews: {((apps['Rating'] == 5.0) & (apps['Reviews_num'] >= 100)).sum()} apps")


We exclude apps with fewer than 100 reviews to compare ratings across categories correctly.
The check showed that 267 of the 271 apps rated 5.0 have fewer than 100 reviews, so the top rating is unreliable.
About 23% of rated apps were excluded.

In [ ]:
# Apps with 100+ reviews and a rating
rating_table = apps[apps["Reviews_num"] >= 100].dropna(subset=["Rating"])
rating_table["Rating"].describe()

In [ ]:
# Share of apps rated 4.5+ and median rating by category
rating_table["Rating_4.5_plus"] = rating_table["Rating"] >= 4.5
top_rating_share = round(rating_table.groupby("Category")["Rating_4.5_plus"].mean()*100)
apps_count = rating_table["Category"].value_counts()

rating = pd.DataFrame({
    "Top_rating_share": top_rating_share,
    "Median_rating": round(rating_table.groupby("Category")["Rating"].median(),1),
    "Apps_count":apps_count
    })

rating.sort_values(["Top_rating_share","Median_rating"],ascending=[False,False])

In [ ]:
# Share of apps rated 4.5+ overall
rating_table["Rating_4.5_plus"].value_counts(normalize=True)*100

**Conclusion 2:**

Among apps with 100+ reviews, about 29% are rated 4.5 or higher.

Among categories with enough apps, BOOKS_AND_REFERENCE (57%) and HEALTH_AND_FITNESS (52%) have the highest share of high ratings.

ENTERTAINMENT has a median rating of 4.2, and only 17% of its apps are rated 4.5+, below the overall 29%. The category seems to stay popular even though users rate its apps lower. This combination of user dissatisfaction and high demand may create an opportunity for a new app.

**Question 3. Do free and paid apps differ in installs and ratings?**

In [ ]:
apps_type = apps["Type"].value_counts(normalize=True)*100
apps_type = apps_type.to_frame(name = "Apps").T
apps_type.plot.barh(stacked=True,
                    title = "Share of free and paid apps",
                    figsize =(9,3),
                    )
plt.xlabel("%")
plt.legend(loc="lower right")
plt.show()

paid_apps = round(apps_type.loc["Apps","Paid"],1)
print(f"Paid apps: {paid_apps}%")

In [ ]:
# Median installs by type
median_by_type = apps.groupby("Type")["Installs_num"].median()

In [ ]:
# Share of apps with 1M+ installs by type
type_share = round(apps.groupby("Type")["Installs_1M_plus"].mean()*100)
installs_by_type = pd.DataFrame({"Installs Median":median_by_type,
                                 "Share_1M_plus, %":type_share })

installs_by_type

Free apps have a 100 times higher median number of installs.

38% of free apps reach 1M+ installs, compared with 3% of paid apps.

In [ ]:
rating_table.head(3)



In [ ]:
# Median rating and share of 4.5+ ratings by type
rating_median = rating_table.groupby("Type")["Rating"].median()
top_rating_share = round(rating_table.groupby("Type")["Rating_4.5_plus"].mean()*100)

rating_by_type = pd.DataFrame({"Rating Median":rating_median,
                               "Share Rating 4.5 plus, %":top_rating_share})

rating_by_type

The median ratings of free and paid apps are almost the same: 4.3 and 4.4.

But 47% of paid apps are rated 4.5+, compared with 28% of free apps.

In [ ]:
# Contingency table: app type vs. rating 4.5+
stat =  pd.crosstab(rating_table["Type"],rating_table["Rating_4.5_plus"])

In [ ]:
stat

In [ ]:
# Chi-square test
from scipy.stats import chi2_contingency
chi2, p, dof, expected = chi2_contingency(stat)
p

The difference is statistically significant (p < 0.001).

**Conclusion 3:**

Free apps get far more installs, while paid apps are associated with higher ratings. For a team that first needs to build an audience, the free model looks more reasonable. However, the data has no revenue information, so which model is more profitable needs a separate check.

**Question 4. What do reviews say: which categories get the most negative feedback?**

In [ ]:
reviews.head(2)

In [ ]:
# Keep only reviews with a sentiment
reviews = reviews.dropna(subset = "Sentiment")

In [ ]:
# Add the app category to each review
merged_table = pd.merge(reviews, apps, on= "App", how = "left")

In [ ]:
# Drop reviews of apps that are not in the main table
merged_table = merged_table.dropna(subset = "Category")

In [ ]:
# Check: reviews after and before the merge
print(len(merged_table))
print(len(reviews))
print(merged_table["Category"].isna().sum())


In [ ]:
# Mark negative reviews
merged_table["Negative_reviews"] = merged_table["Sentiment"] == "Negative"

In [ ]:
# Share of negative reviews by category
n_reviews_share = round(merged_table.groupby("Category")["Negative_reviews"].mean()*100)
count_reviews = merged_table["Category"].value_counts()


negative_reviews_by_category = pd.DataFrame({"Share negative reviews, %":n_reviews_share,
                   "Count reviews":count_reviews})

negative_reviews_by_category.sort_values("Share negative reviews, %",ascending = False)

In [ ]:
plt.figure(figsize=(10, 15))

sort_share_categories = negative_reviews_by_category.sort_values(
    "Share negative reviews, %", ascending=False
).index

sns.barplot(
    data=merged_table,
    x="Negative_reviews",
    y="Category",
    order=sort_share_categories,
    errorbar=None
).set(
    title="Share of negative reviews by category",
    xlabel="Share of negative reviews",
    ylabel="Category"
)

plt.show()

In [ ]:
# Baseline: share of negative reviews across all categories
merged_table["Negative_reviews"].value_counts(normalize=True)*100

**Conclusion 4:**

GAME has the highest share of negative reviews: 37% vs 22% across all reviews.

ENTERTAINMENT ranks 5th (25%), but only 3 percentage points above the baseline. This matches its lower ratings in Question 2, but the signal is weak.

Limitation: reviews are available only for some apps; about 4% of reviews were excluded because their apps are not in the main table.

## 7. Conclusions

ENTERTAINMENT looks like the most promising category: 84% of its apps have 1M+ installs, and competition is low: it ranks 24th of 33 by number of apps.

The share of ENTERTAINMENT apps rated 4.5+ is lower than among all apps with 100+ reviews: 17% vs 29%. Negative reviews make up 25%, slightly above the 22% across all reviews.

This suggests that ENTERTAINMENT apps are popular despite user dissatisfaction. High demand combined with lower ratings means that a high-quality app could stand out.

The free model is associated with a larger audience: a median of 100,000 installs vs 1,000 for paid apps. Paid apps are more often highly rated: 47% vs 28% (p < 0.001). For a team that first needs to build an audience, the free model looks more reasonable, but the data does not show which option is more profitable.

### Decisions made during the analysis
- Apps listed in several categories kept only one category after removing duplicates
- Missing ratings were not filled, because invented ratings would distort the results
- Ratings were analyzed only for apps with 100+ reviews (~23% of rated apps excluded)

## 8. Next Steps
- **For the team:** consider ENTERTAINMENT for a pilot app with a free model
- **To validate:** collect fresh data, plus data on revenue and user retention
- **To go deeper:** find out which ENTERTAINMENT apps are the most successful and what users criticize in their reviews